<a href="https://colab.research.google.com/github/Zuhair0000/TensorFlow-ML-DL-Project-Practice/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

# Generate a synthetic imbalanced dataset (like fraud)
X, y = make_classification(n_samples=10000, n_features=10, n_classes=2, weights=[0.95, 0.05], random_state=42)
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(10)])
df['target'] = y

# Inject some fake missing values (-1) to simulate real-world messy data
df.loc[np.random.choice(df.index, 500), 'feature_3'] = -1
df.to_csv('dataset.csv', index=False)
print("dataset.csv created successfully!")

dataset.csv created successfully!


# Load Data with Polars

In [13]:
import polars as pl

In [14]:
lf = pl.scan_csv('dataset.csv')

In [15]:
clean_lf = lf.with_columns(
    pl.when(pl.col('feature_3') == -1).then(None).otherwise('feature_3').alias('feature_3')
)

In [16]:
df = clean_lf.collect().to_pandas()

In [17]:
X = df.drop(columns=['target'])
y = df['target']

In [20]:
X.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9
0,1.195860,0.357385,-0.665862,-0.492845,-0.863395,-0.050796,-1.933989,2.081684,0.041266,-0.258298
1,0.825314,1.109459,-0.323136,-1.419545,-1.593368,1.107295,1.243526,-0.172200,1.150359,0.147744
2,-0.906713,-0.593314,0.710714,-1.255291,-0.850725,-0.317640,1.499045,0.434477,0.423678,1.251380
3,1.200848,-1.306530,-0.496291,-1.858754,-2.127356,-0.879635,-0.393494,-0.101213,-1.624066,0.443553
4,-1.204798,0.078464,0.705181,0.224765,0.618707,1.534946,-0.302288,2.325055,0.495505,0.538133


# Train/Test Split

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

num_cols = X_train.columns.to_list()

num_enc = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_enc, num_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# ML

In [22]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50, class_weight='balanced', random_state=42)
rf.fit(X_train_processed, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=50,
                       random_state=42)

In [23]:
from sklearn.metrics import classification_report

y_pred = rf.predict(X_test_processed)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1890
           1       0.84      0.49      0.62       110

    accuracy                           0.97      2000
   macro avg       0.91      0.74      0.80      2000
weighted avg       0.96      0.97      0.96      2000



# DL (TF)

In [24]:
import tensorflow as tf

train_ds = tf.data.Dataset.from_tensor_slices((X_train_processed, y_train)).shuffle(buffer_size=1024).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices((X_test_processed, y_test)).batch(64)

In [26]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_processed.shape[1], )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy'
)

model.fit(train_ds, epochs=5, batch_size=64)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1946
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1242
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1203
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1172
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.1165


In [29]:
tf_outputs = model.predict(X_test_processed)
y_pred_tf = (tf_outputs > 0.5).astype(int)

print(classification_report(y_test, y_pred_tf))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1890
           1       0.67      0.46      0.55       110

    accuracy                           0.96      2000
   macro avg       0.82      0.73      0.76      2000
weighted avg       0.95      0.96      0.95      2000



# DL (Pytorch)

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64, shuffle=False)

In [33]:
class FraudNet(nn.Module):
  def __init__(self, input_dim):
    super(FraudNet, self).__init__()

    self.layer1 = nn.Linear(input_dim, 16)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(16, 1)

  def forward(self, x):
    x = self.relu(self.layer1(x))
    x = self.layer2(x)

    return x

In [34]:
model = FraudNet(input_dim=X_train_processed.shape[1])

criterion = nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(), lr=0.01)

In [36]:
epochs = 5

for epoch in range(epochs):
  model.train()

  for batch_X, batch_y in train_loader:
    optimizer.zero_grad()

    outputs = model(batch_X)

    loss = criterion(outputs, batch_y)

    loss.backward()

    optimizer.step()
  print(f"Ecpoch {epoch+1}/{epochs}")

Ecpoch 1/5
Ecpoch 2/5
Ecpoch 3/5
Ecpoch 4/5
Ecpoch 5/5


In [38]:
model.eval()

with torch.no_grad():
  test_outputs = model(X_test_tensor)
  pt_pred = (torch.sigmoid(test_outputs)>0.5).float()

print(classification_report(y_test_tensor, pt_pred))

              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      1890
         1.0       0.71      0.45      0.55       110

    accuracy                           0.96      2000
   macro avg       0.84      0.72      0.76      2000
weighted avg       0.95      0.96      0.96      2000

